# How much performance do you lose by using smaller Transformers?

Exploratory companion to the scripted pipeline. Run the benchmark first:

```bash
python run_benchmark.py --num-threads 12   # check_env.py prints the right value for your machine
```

This notebook reads `results/benchmark.csv` and `results/analysis.json` — it does not retrain
anything, so it is cheap to re-run and safe to edit.

It goes past the report in three places:

1. **Scaling fit** — accuracy against encoder size on a log axis, so the cost of shrinking is
   expressed as points per halving rather than as an unordered list of models.
2. **Latency model** — checking that measured latency tracks the analytic FLOP proxy
   `layers x hidden^2`, which is what lets you predict a model you have not run.
3. **Budget queries** — given a latency ceiling, which model maximises accuracy.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "report.py").exists():
    ROOT = ROOT.parent          # launched from notebooks/
sys.path.insert(0, str(ROOT))

import plotting

RESULTS = ROOT / "results"
df = pd.read_csv(RESULTS / "benchmark.csv")
analysis = json.loads((RESULTS / "analysis.json").read_text(encoding="utf-8"))
rows = analysis["rows"]
summary = analysis["summary"]

print(f"{summary['dataset']}  |  baseline {summary['baseline_label']}  |  "
      f"{summary['eval_examples']:,} eval examples")
df[["label", "accuracy", "train_time_s", "latency_ms_p50", "params_total", "size_on_disk_mb"]]

## The table, as the report renders it

In [ ]:
view = pd.DataFrame({
    "Model": df["label"],
    "Accuracy (%)": (100 * df["accuracy"]).round(2),
    "Acc retained (%)": (100 * df["accuracy_retention"]).round(2),
    "Delta (pp)": df["delta_accuracy_pp"].round(2),
    "McNemar p": df["mcnemar_p"].round(4),
    "Train (s)": df["train_time_s"].round(0),
    "p50 (ms)": df["latency_ms_p50"].round(1),
    "Speedup": df["latency_speedup"].round(2),
    "Params (M)": (df["params_total"] / 1e6).round(1),
})
view.set_index("Model")

## Figures, inline

Same builders the report uses, so the notebook cannot drift from `plots/`.

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

theme = plotting.THEMES["light"]        # switch to "dark" for a dark notebook
plotting.apply_theme(theme)
frontier = set(summary["pareto_frontier"])

fig = plotting.fig_pareto(rows, frontier, theme,
                          latency_key="latency_ms_p50",
                          latency_label="Latency, batch = 1  (ms, median)")
fig

In [ ]:
plotting.apply_theme(theme)
fig = plotting.fig_cost_panels(rows, theme,
                               latency_key="latency_ms_p50",
                               latency_label="Latency, batch = 1  (ms, median)")
fig

## 1. Scaling: what does halving the encoder cost?

Accuracy is fitted against $\log_2$ of the **non-embedding** parameter count. The embedding
table is a lookup, not compute — BERT-base and DistilBERT carry an identical ~23.8M-parameter
embedding, so a fit on total parameters would attribute to depth an effect that depth does
not have.

The slope is read directly as **accuracy points per doubling of encoder size**. It is a local
linearisation of a curve that is certainly not linear over many orders of magnitude, and with
a handful of points it is descriptive, not predictive — the residuals below say how much to trust it.

In [ ]:
x = np.log2(df["params_non_embedding"].to_numpy(dtype=float))
y = 100 * df["accuracy"].to_numpy(dtype=float)

if len(x) >= 2:
    slope, intercept = np.polyfit(x, y, 1)
    resid = y - (slope * x + intercept)
    ss_tot = ((y - y.mean()) ** 2).sum()
    r2 = 1 - (resid ** 2).sum() / ss_tot if ss_tot > 0 else float("nan")

    print(f"slope      {slope:+.3f} accuracy points per doubling of encoder parameters")
    print(f"halving    {-slope:+.3f} points lost per halving")
    print(f"R^2        {r2:.3f}   (n = {len(x)} models)")
    print(f"max |resid| {np.abs(resid).max():.3f} pp")
else:
    print("Need at least two models. Run the sweep with --models bert-base distilbert minilm-l12")

In [ ]:
import matplotlib.pyplot as plt

plotting.apply_theme(theme)
fig, ax = plt.subplots(figsize=(6.6, 4.0), constrained_layout=True)
colors = plotting.series_colors(len(df), theme)

if len(x) >= 2:
    grid = np.linspace(x.min() - 0.3, x.max() + 0.3, 100)
    ax.plot(2 ** grid / 1e6, slope * grid + intercept,
            color=theme["axis"], linewidth=2.0, zorder=2,
            label=f"least squares: {slope:+.2f} pp / doubling")

for (_, row), color in zip(df.iterrows(), colors):
    ax.scatter(row["params_non_embedding"] / 1e6, 100 * row["accuracy"],
               s=190, color=color, edgecolor=theme["surface"], linewidth=2.0, zorder=4)
    ax.annotate(row["label"], xy=(row["params_non_embedding"] / 1e6, 100 * row["accuracy"]),
                xytext=(0, 18), textcoords="offset points", ha="center",
                fontsize=9, color=theme["ink"])

ax.set_xscale("log", base=2)
ax.set_xlabel("Non-embedding parameters  (millions, log2)")
ax.set_ylabel("Accuracy  (%)")
ax.set_title("Accuracy against encoder size")
ax.legend(loc="lower right", labelcolor=theme["ink2"], fontsize=9)
plotting._style_axes(ax, theme, value_axis="y")
fig

## 2. Does measured latency follow the FLOP proxy?

A transformer encoder's forward cost at fixed sequence length is dominated by the projection
and feed-forward matmuls, which scale as $L \cdot d^2$ for $L$ layers of width $d$. If measured
latency is proportional to that proxy, then the cost of an unrun configuration is predictable
from its config alone.

Where the ratio drifts, the gap is overhead that does not scale with $d^2$ — Python dispatch,
kernel launches, memory traffic — and it is exactly that overhead that makes small models less
fast than their parameter count promises.

In [ ]:
arch = pd.DataFrame([r["architecture"] for r in rows], index=[r["label"] for r in rows])
proxy = arch["num_layers"] * arch["hidden_size"] ** 2

compare = pd.DataFrame({
    "layers": arch["num_layers"],
    "hidden": arch["hidden_size"],
    "L*d^2 (rel.)": (proxy / proxy.max()).round(3),
    "p50 latency (rel.)": (df["latency_ms_p50"].to_numpy() / df["latency_ms_p50"].max()).round(3),
})
compare["measured / predicted"] = (
    compare["p50 latency (rel.)"] / compare["L*d^2 (rel.)"]
).round(2)
compare

A `measured / predicted` ratio above 1 means the model is **slower than its arithmetic**:
the fixed per-layer overhead has not shrunk with the width. That is the usual reason a model
with a quarter of the FLOPs does not run four times faster.

## 3. Budget queries

The form the decision actually takes: a latency ceiling comes from the product, and the
question is what accuracy is reachable underneath it.

In [ ]:
def best_under(budget_ms: float) -> str:
    fits = df[df["latency_ms_p50"] <= budget_ms]
    if fits.empty:
        return f"{budget_ms:>6.0f} ms   nothing in the sweep fits"
    pick = fits.loc[fits["accuracy"].idxmax()]
    return (f"{budget_ms:>6.0f} ms   {pick['label']:<14} "
            f"{100 * pick['accuracy']:.2f}%  ({100 * pick['accuracy_retention']:.1f}% of baseline)")

ceiling = df["latency_ms_p50"].max()
for budget in [ceiling * f for f in (0.25, 0.4, 0.6, 0.8, 1.0)]:
    print(best_under(budget))

In [ ]:
# Same question in throughput terms: cost to score a corpus, relative to the baseline.
CORPUS = 10_000_000
hours = CORPUS / df["throughput_examples_per_s"] / 3600
base_hours = hours[df["is_baseline"].astype(bool)].iloc[0]
cost = pd.DataFrame({
    "Model": df["label"],
    "Throughput (ex/s)": df["throughput_examples_per_s"].round(1),
    f"Hours for {CORPUS:,} docs": hours.round(1),
    "Hours saved vs baseline": (base_hours - hours).round(1),
    "Accuracy given up (pp)": (-df["delta_accuracy_pp"]).round(2) + 0.0,  # + 0.0: no "-0.00"
}).set_index("Model")
cost

## Caveats worth keeping in view

- **One task.** Topic classification on short news text is close to the easiest thing a BERT
  encoder does; the headroom above a bag-of-words baseline is small, so the *absolute* gap
  between a large and a small encoder is compressed. Expect wider gaps on NLI, QA, or
  anything needing multi-sentence reasoning.
- **One sequence length.** Attention is quadratic in length while the feed-forward is linear,
  so the speed ranking can move with `--max-length`.
- **Sample size bounds resolution.** For a paired comparison the tightest resolvable gap is
  roughly $2\sqrt{d/n}$, where $d$ is the share of examples on which the two models disagree.
  Here $d$ is 3–6%, so at n = 2000 that is about 1 pp. Use the McNemar column, not the
  ranking, and raise `--max-eval-samples` to resolve smaller differences.
- **One seed drives the figures.** Fine-tuning variance on runs this size is a few tenths of
  a point. Run `--seeds 42 43 44` before believing a sub-half-point difference.
- **One machine.** Latency, throughput and training time are measured on the hardware that
  ran the sweep; see the README for why the speed ratios, not just the absolute times, can
  change on another machine.